# Drafting Demo: Counter-pick assistant (advanced)

This notebook implements an improved drafting algorithm with role-awareness, synergy, and evaluation utilities. It keeps the same safety rule: do not store your Riot API key in the repo—use the provided fetch script and set the `RIOT_API_KEY` environment variable.

In [1]:
# Standard imports
import os
from pathlib import Path
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Data loading helpers
This section attempts to load either a processed `matches.csv` or the per-champion summary CSVs that are included in the repo.

In [2]:
BASE = Path('Winrate_Prediction')
PROC_DIR = BASE / 'data' / 'processed'
AO_DIR = BASE / 'analysis_outputs'
matches_df = None
# Try common processed filenames first
for cand in ['matches.csv','processed_matches.csv']:
            p = PROC_DIR / cand
            if p.exists():
                matches_df = pd.read_csv(p)
                print('Loaded', p)
                break
# fallback to per-role summaries
champ_stats = None
if matches_df is None and AO_DIR.exists():
    csvs = sorted(glob.glob(str(AO_DIR / '*_per_champion_stats.csv')) )
    if csvs:
        frames = [pd.read_csv(x) for x in csvs]
        champ_stats = pd.concat(frames, ignore_index=True, sort=False)
        # heuristically find champion name and win columns
        name_cols = [c for c in champ_stats.columns if 'champ' in c.lower()]
        win_cols = [c for c in champ_stats.columns if 'win' in c.lower()]
        if name_cols and win_cols:
            champ_stats = champ_stats[[name_cols[0], win_cols[0]]].rename(columns={name_cols[0]:'champion', win_cols[0]:'win_rate'}).drop_duplicates(subset=['champion']).set_index('champion')
        print('Loaded champion summaries from analysis_outputs')

print('matches_df available:', matches_df is not None)
print('champ_stats available:', champ_stats is not None)

matches_df available: False
champ_stats available: False


In [3]:
# Consolidate role assignments from per-role summary CSVs.
# For each *_per_champion_stats.csv file, we treat the filename prefix as the role
# (e.g., Top_per_champion_stats.csv -> 'Top') and pick for each champion the role
# where it has the highest play count.
role_map = {}
role_pools_from_map = {}
if AO_DIR.exists():
    per_role_files = sorted(glob.glob(str(AO_DIR / '*_per_champion_stats.csv')))
    counts = {}
    for f in per_role_files:
        role = Path(f).name.split('_')[0]
        try:
            df_r = pd.read_csv(f)
        except Exception:
            continue
        # find champion name column heuristically
        name_col = None
        for c in df_r.columns:
            if 'name' in c.lower() or 'champ' in c.lower():
                name_col = c
                break
        # find a count column
        count_col = None
        for c in ['count','games','plays','games_played','play_count']:
            if c in df_r.columns:
                count_col = c
                break
        for _, r in df_r.iterrows():
            if name_col is None:
                continue
            champ = str(r[name_col])
            cnt = int(r[count_col]) if count_col and pd.notna(r.get(count_col)) else 1
            counts.setdefault(champ, {}).setdefault(role, 0)
            counts[champ][role] += cnt
    # choose best role per champion
    for champ, rc in counts.items():
        best_role = max(rc.items(), key=lambda x: x[1])[0]
        role_map[champ] = best_role
        role_pools_from_map.setdefault(best_role, []).append(champ)
    # ensure an 'any' pool
    role_pools_from_map['any'] = sorted(list({c for chs in role_pools_from_map.values() for c in chs}))
print('Inferred roles for', len(role_map), 'champions. Sample:', list(role_map.items())[:10])

Inferred roles for 0 champions. Sample: []


## Per-role summary tables
The repository provides per-role summary CSVs under `Winrate_Prediction/analysis_outputs/`.
The cell below loads those files, normalises common column names, and shows a combined table
grouped by role so you can inspect corrected role assignments and top champions per role.

In [4]:
# Display per-role summary CSVs (if present) in a clear table
from IPython.display import display, HTML
per_role_files = sorted(glob.glob(str(AO_DIR / '*_per_champion_stats.csv')))
dfs = []
for f in per_role_files:
    role = Path(f).name.split('_')[0]
    try:
        df = pd.read_csv(f)
    except Exception as e:
        print('Could not read', f, '->', e)
        continue
    # heuristics for columns
    name_col = next((c for c in df.columns if 'name' in c.lower() or 'champ' in c.lower()), None)
    games_col = next((c for c in df.columns if any(k in c.lower() for k in ['count','games','plays','games_played','play_count'])), None)
    win_col = next((c for c in df.columns if 'win' in c.lower()), None)
    if name_col is None:
        continue
    cols = [name_col]
    if games_col: cols.append(games_col)
    if win_col: cols.append(win_col)
    df2 = df[cols].copy()
    rename_map = {name_col: 'champion'}
    if games_col: rename_map[games_col] = 'games'
    if win_col: rename_map[win_col] = 'win_rate'
    df2 = df2.rename(columns=rename_map)
    df2['role'] = role
    dfs.append(df2)
if dfs:
    combined = pd.concat(dfs, ignore_index=True, sort=False)
    combined['champion'] = combined['champion'].astype(str)
    # show a sorted, concise table
    sort_cols = ['role'] + (["games"] if 'games' in combined.columns else [])
    display(combined.sort_values(by=sort_cols, ascending=[True]+([False] if 'games' in combined.columns else [])).reset_index(drop=True))
else:
    print('No per-role summary CSVs found in', AO_DIR)

No per-role summary CSVs found in Winrate_Prediction\analysis_outputs


In [5]:
# Also show the inferred role map (from earlier consolidation) so reviewers can verify fixes
if 'role_map' in globals() and role_map:
    rm_df = pd.DataFrame(list(role_map.items()), columns=['champion','inferred_role']).sort_values(['inferred_role','champion']).reset_index(drop=True)
    display(rm_df.head(200))
else:
    print('role_map not found (run the consolidation cell)')
# summary of role pools sizes
if 'role_pools_from_map' in globals():
    print('Role pools sizes:')
    for r,v in sorted(role_pools_from_map.items()):
        print(f'  {r}: {len(v)} champs')

role_map not found (run the consolidation cell)
Role pools sizes:


## Pairwise and synergy matrices
We attempt to compute two matrices if possible from `matches_df`:
- `pairwise_win`: estimated probability champion A beats champion B (A vs B)
- `synergy`: estimated lift when two champions are on the same team (co-play win rate / baseline)
Both are built with fallbacks to champion-level win rates when per-match detail is unavailable.

In [6]:
def build_pairwise_and_synergy(matches):
    # Expect matches to have `participants` structure OR per-player rows. We'll support
    # the common processed schema where each row is a participant with 'champion',
    # 'win' and 'match_id' and optionally 'team' and 'role'.
    if matches is None:
        return None, None
    df = matches.copy()
    cols = [c.lower() for c in df.columns]
    # find participant-style rows: champion + win + match_id
    if 'champion' in df.columns and 'win' in [c.lower() for c in df.columns] and 'match_id' in [c.lower() for c in df.columns]:
        # make sure columns exist with standard names
        champ_col = [c for c in df.columns if c.lower()=='champion'][0]
        win_col = [c for c in df.columns if c.lower()=='win'][0]
        mid_col = [c for c in df.columns if c.lower()=='match_id'][0]
        # Pairwise: we need A vs B pairs. Build by grouping participants per match
        # and creating cross pairs between teams.
        # First, attempt to infer team side column (0/1 or 'team')
        team_col = None
        for c in df.columns:
            if 'team' in c.lower():
                team_col = c
                break
        # create team mapping per match
        pairwise_rows = []
        synergy_rows = []
        for mid, group in df.groupby(mid_col):
            # group should contain participants for the match
            if team_col is None:
                # heuristics: assume first 5 vs last 5
                g = group.reset_index(drop=True)
                if len(g) < 10:
                    continue
                teamA = g.iloc[:len(g)//2]
                teamB = g.iloc[len(g)//2:]
            else:
                teamA = group[group[team_col]==group[team_col].unique()[0]]
                teamB = group[group[team_col]!=group[team_col].unique()[0]]
            # compute pairwise A vs B entries
            for _, a in teamA.iterrows():
                for _, b in teamB.iterrows():
                    pairwise_rows.append({'a': a[champ_col], 'b': b[champ_col], 'a_win': float(a[win_col])})
                    pairwise_rows.append({'a': b[champ_col], 'b': a[champ_col], 'a_win': float(b[win_col])})
            # synergy: pairs on same team
            for x in [teamA, teamB]:
                comps = x[champ_col].tolist()
                # win flag for that team (majority) approximate by participants' win value mode
                team_win = x[win_col].mode()[0] if not x[win_col].mode().empty else None
                for i in range(len(comps)):
                    for j in range(i+1, len(comps)):
                        synergy_rows.append({'c1': comps[i], 'c2': comps[j], 'team_win': float(team_win) if team_win is not None else np.nan})
        # aggregate pairwise and synergy
        pair_df = pd.DataFrame(pairwise_rows)
        if not pair_df.empty:
            grouped = pair_df.groupby(['a','b'])['a_win'].agg(['mean','count']).reset_index().rename(columns={'mean':'win_rate','count':'n'})
            pairwise = grouped.pivot(index='a', columns='b', values='win_rate')
        else:
            pairwise = pd.DataFrame()
        syn_df = pd.DataFrame(synergy_rows)
        if not syn_df.empty:
            syng = syn_df.groupby(['c1','c2'])['team_win'].agg(['mean','count']).reset_index().rename(columns={'mean':'win_rate','count':'n'})
            # make symmetric synergy matrix (c1,c2 and c2,c1 same)
            syn_pivot = syng.pivot(index='c1', columns='c2', values='win_rate')
            syn_full = syn_pivot.copy()
            for r in syn_full.index:
                for c in syn_full.columns:
                    if pd.isna(syn_full.loc[r, c]):
                        if c in syn_pivot.index and r in syn_pivot.columns:
                            val = syn_pivot.loc[c, r]
                            syn_full.loc[r, c] = val if not pd.isna(val) else np.nan
            synergy = syn_full
        else:
            synergy = pd.DataFrame()
        return pairwise, synergy
    else:
        return None, None

In [7]:
pairwise_win, synergy = build_pairwise_and_synergy(matches_df)
print('pairwise_win available:', pairwise_win is not None and not pairwise_win.empty)
print('synergy available:', synergy is not None and not synergy.empty)

pairwise_win available: False
synergy available: False


## Role-aware drafting and scoring
We'll implement a scoring function that combines: (1) counter score vs enemy picks using `pairwise_win`, (2) synergy bonus between selected allies using `synergy`, and (3) role coverage and diversity penalties. Then we'll provide a greedy+beam search picker that respects role constraints and bans.

In [8]:
# Utilities to get champion pools per role from matches_df or fallback
def champion_role_pools(matches):
    # expect a column like 'role' or 'lane' in participant rows; otherwise return broad pool
    if matches is None:
        return {}
    cols = [c.lower() for c in matches.columns]
    role_col = None
    for c in matches.columns:
        if 'role' in c.lower() or 'lane' in c.lower():
            role_col = c
            break
    pools = {}
    if role_col is None:
        pools['any'] = sorted(list(set(matches['champion'].astype(str).tolist())))
        return pools
    for r, g in matches.groupby(role_col):
        pools[r] = sorted(list(g['champion'].astype(str).unique()))
    return pools

role_pools = champion_role_pools(matches_df)
print('Detected role pools keys:', list(role_pools.keys())[:10])

Detected role pools keys: []


In [9]:
# If we couldn't infer role pools from matches_df, fall back to inferred per-role mapping
if (not role_pools or len(role_pools) == 0) and 'role_pools_from_map' in globals():
    role_pools = role_pools_from_map
    print('Using role pools inferred from per-role summary CSVs:', list(role_pools.keys())[:10])
else:
    print('Using role pools derived from match participant data (if available).')

Using role pools inferred from per-role summary CSVs: []


In [10]:
def team_score(team, enemy_picks, pairwise, synergy, role_map=None, w_counter=0.7, w_synergy=0.25, w_diversity=0.05):
    # team: list of champion names (strings)
    # enemy_picks: list of enemy champion strings
    team = [str(x) for x in team]
    enemy_picks = [str(x) for x in enemy_picks]
    # Counter component: mean estimated winrate of our champs vs each enemy champion (averaged)
    counter_vals = []
    if pairwise is not None and not pairwise.empty:
        for t in team:
            vals = []
            for e in enemy_picks:
                if e in pairwise.columns and t in pairwise.index:
                    v = pairwise.loc[t,e]
                    if pd.notna(v):
                        vals.append(float(v))
            counter_vals.append(np.mean(vals) if vals else 0.5)
    else:
        # fallback to 0.5 baseline if no data
        counter_vals = [0.5]*len(team)
    counter_score = float(np.mean(counter_vals))
    # Synergy component: average synergy between every pair in team (higher is better)
,
,

''